In [1]:
import os
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'  # force synchronous debugging

from datetime import datetime
from pathlib import Path
import logging
from huggingface_hub import login, whoami
from transformers.utils.logging import disable_progress_bar

from gsm_benchmarker.dataset_wrapper import GSMSymbolicDataset
from gsm_benchmarker.benchmark_config import BenchmarkConfig
from gsm_benchmarker.benchmark import BenchmarkRunner
from gsm_benchmarker.models_config_parser import ModelsConfig
from gsm_benchmarker.utils.logging_setup import install_colored_logger, setup_log_file_handler
from gsm_benchmarker.utils.seeds import set_seed


disable_progress_bar()
set_seed(42)


hf_api_token = os.environ['HUGGINGFACEHUB_API_TOKEN']
login(hf_api_token)

whoami()['name']

openai package not installed; OpenAI models will not be available
anthropic package not installed; Anthropic models will not be available
google-genai package not installed; Gemini family models will not be available


'the-mysh'

In [2]:
OUTPUT_ROOT_PATH = Path(f"../../../data/gsm-symbolic").resolve()
RESULTS_ROOT_PATH = OUTPUT_ROOT_PATH / f"outputs/{datetime.now().strftime('%Y%m%d_%H%M%S')}"
print(OUTPUT_ROOT_PATH)
print(RESULTS_ROOT_PATH)
print(os.environ['HF_HOME'])

/home/guests2/dkd/data/gsm-symbolic
/home/guests2/dkd/data/gsm-symbolic/outputs/20251029_170454
/media/generalstorage4/dkdstorage


In [3]:
for log_name in ('urllib3', 'fsspec', 'filelock', 'h5py', 'httpcore', 'httpx', 'google_genai', 'jax', 'root', 'bitsandbytes', 'transformers_modules'):
    logging.getLogger(log_name).setLevel(logging.WARNING)

install_colored_logger(level=logging.DEBUG)
setup_log_file_handler(OUTPUT_ROOT_PATH / "logs")

gsm_benchmarker.utils.logging_setup [INFO] 2025-10-29 17:04:54: Python logs will be stored to file: /home/guests2/dkd/data/gsm-symbolic/logs/20251029-170454.log


In [4]:
models_config = ModelsConfig()
all_models = models_config.all_models
for m in all_models:
    print(m.name, m.api_type, m.extra_kwargs)  # note: openai - insufficient quota for benchmarking

google/gemma-2b None {}
google/gemma-2b-it None {}
google/gemma-7b None {}
google/gemma-7b-it None {}
google/gemma-2-2b None {}
google/gemma-2-2b-it None {}
google/gemma-2-9b None {}
google/gemma-2-9b-it None {}
google/gemma-2-27b-it None {}
microsoft/phi-2 None {}
microsoft/Phi-3-mini-128k-instruct None {}
microsoft/Phi-3-small-128k-instruct None {'from_pretrained': {'revision': 'ad85cab62be398dc90203c4377a4ccbf090fbb36'}, 'tokeniser_from_pretrained': {'revision': 'ad85cab62be398dc90203c4377a4ccbf090fbb36', 'clean_up_tokenization_spaces': True}}
microsoft/Phi-3-medium-128k-instruct None {}
microsoft/Phi-3.5-mini-instruct None {}
mistralai/Mistral-7B-v0.1 None {}
mistralai/Mistral-7B-Instruct-v0.1 None {}
mistralai/Mistral-7B-v0.3 None {}
mistralai/Mistral-7B-Instruct-v0.3 None {}
mistralai/Mathstral-7B-v0.1 None {}
meta-llama/Meta-Llama-3-8B None {}
meta-llama/Meta-Llama-3-8B-Instruct None {}
gpt-4o-mini APIType.openai {}
gpt-4o APIType.openai {}
o1-mini APIType.openai {}
o1-preview A

In [5]:
open_models = models_config.open_models
print([m.name for m in open_models])

['google/gemma-2b', 'google/gemma-2b-it', 'google/gemma-7b', 'google/gemma-7b-it', 'google/gemma-2-2b', 'google/gemma-2-2b-it', 'google/gemma-2-9b', 'google/gemma-2-9b-it', 'google/gemma-2-27b-it', 'microsoft/phi-2', 'microsoft/Phi-3-mini-128k-instruct', 'microsoft/Phi-3-small-128k-instruct', 'microsoft/Phi-3-medium-128k-instruct', 'microsoft/Phi-3.5-mini-instruct', 'mistralai/Mistral-7B-v0.1', 'mistralai/Mistral-7B-Instruct-v0.1', 'mistralai/Mistral-7B-v0.3', 'mistralai/Mistral-7B-Instruct-v0.3', 'mistralai/Mathstral-7B-v0.1', 'meta-llama/Meta-Llama-3-8B', 'meta-llama/Meta-Llama-3-8B-Instruct', 'gpt-4o-mini', 'gpt-4o', 'o1-mini', 'o1-preview']


In [6]:
variants = GSMSymbolicDataset.Variant

In [7]:

# br = BenchmarkRunner(
#     models=open_models[9:],
#     dset_variants=[variants.GSM8K, variants.main, variants.p1, variants.p2],
#     storage_path==RESULTS_ROOT_PATH,
#     config=BenchmarkConfig(trust_remote_code=True, gpu0_max_memory="14GB", cpu_max_memory="10GB")
# )

In [8]:
# br.run(n_sets=2, n_per_set=2)

In [9]:
failed_models = [models_config['microsoft/Phi-3-small-128k-instruct']]  #, 'mistralai/Mistral-7B-v0.3', 'mistralai/Mistral-7B-Instruct-v0.3']
br2 = BenchmarkRunner(
    models=failed_models,
    dset_variants=[variants.GSM8K, variants.main, variants.p1, variants.p2],
    storage_path=RESULTS_ROOT_PATH,
    config=BenchmarkConfig(trust_remote_code_global=True, gpu0_max_memory="46GiB", cpu_max_memory="100GiB")
)
br2.run(n_sets=2, n_per_set=2)

gsm_benchmarker.utils.path_ops [DEBUG] 2025-10-29 17:04:55: Creating directories at /home/guests2/dkd/data/gsm-symbolic/outputs/20251029_170454
gsm_benchmarker.benchmark [INFO] 2025-10-29 17:04:55: ========== Evaluating model 1/1: microsoft/Phi-3-small-128k-instruct ==========
gsm_benchmarker.model_evaluator [DEBUG] 2025-10-29 17:04:55: Initialising a HuggingFace model
gsm_benchmarker.hf_model_wrapper [INFO] 2025-10-29 17:04:55: Setting up model microsoft/Phi-3-small-128k-instruct
gsm_benchmarker.hf_model_wrapper [DEBUG] 2025-10-29 17:04:55: Loading tokeniser
gsm_benchmarker.hf_model_wrapper [DEBUG] 2025-10-29 17:04:55: Passing extra kwargs to 'AutoTokenizer.from_pretrained': {'revision': 'ad85cab62be398dc90203c4377a4ccbf090fbb36', 'clean_up_tokenization_spaces': True}
gsm_benchmarker.hf_model_wrapper [DEBUG] 2025-10-29 17:04:56: CUDA available
gsm_benchmarker.hf_model_wrapper [DEBUG] 2025-10-29 17:04:56: Loading model with CUDA
gsm_benchmarker.hf_model_wrapper [DEBUG] 2025-10-29 17:04

KeyboardInterrupt: 

In [ ]:
print(br2.summarise_failures())

In [ ]:
br.summarise_results()

In [ ]:
el = br.results[GSMSymbolicDataset.Variant.main][open_models[-1]].loc[0, 0]
el

In [ ]:
print(el.question)
print()
print(el.response)